In [1]:
import sys
sys.path.append('../externals/DynaMix-python')

import numpy as np
import pandas as pd
import pathlib
import matplotlib.pyplot as plt
from neurodsp.spectral import compute_spectrum
from neurodsp.plts.spectral import plot_power_spectra
import os
from joblib import Parallel, delayed
from tqdm import tqdm

from src.dynamix.model.forecaster import DynaMixForecaster
from src.dynamix.utilities.plotting_eval import plot_TS_forecast, plot_3D_attractor
from src.dynamix.utilities.utilities import load_hf_model
import torch

In [2]:
# define file paths
eeg_path = '/oscar/data/sjones/shared/TDBRAIN_preprocessed/preprocessed'
metadata_path = '/oscar/home/scbao/tdbrain-model/data/TDBRAIN_participants_V2.tsv'
subj_list = os.listdir(eeg_path)
print("Number of total subjects:", len(subj_list))

# read data
metadata_df = pd.read_csv(metadata_path, delimiter='\t')

Number of total subjects: 1274


In [3]:
# generate mask to pull out MDD, DISC subjects only
subj_mask = np.isin(metadata_df['participants_ID'].values, subj_list)
discovery_mask = metadata_df['DISC/REP'].values == 'DISCOVERY'
dataset_mask = metadata_df['Dataset'].values == 'MDD-rTMS'
rTMS_mask = ~metadata_df['rTMS PROTOCOL'].isna() # excludes 1 participant

mask = np.logical_and.reduce([subj_mask, discovery_mask, dataset_mask, rTMS_mask])

df = metadata_df[mask].copy() # .copy because plan to add columns later
print("Shape after keeping MDD-rTMS, Discovery rows:", df.shape)

Shape after keeping MDD-rTMS, Discovery rows: (131, 111)


In [6]:
duplicate_ids = df["participants_ID"].value_counts()
duplicate_ids = duplicate_ids[duplicate_ids > 1].index
print(duplicate_ids)

df[df["participants_ID"] == 'sub-88012461']

Index(['sub-88012461', 'sub-88009901', 'sub-88006161', 'sub-88010981',
       'sub-88023125', 'sub-88022089', 'sub-88017409', 'sub-88045713'],
      dtype='object', name='participants_ID')


,participants_ID,DISC/REP,indication,formal_status,Dataset,Consent,sessSeason,sessTime,Responder,Remitter,...,BDI_post,rTMS PROTOCOL,ADHD_pre_Hyp_leading,ADHD_pre_Att_leading,ADHD_post_Att_leading,ADHD_post_Hyp_leading,NF Protocol,YBOCS_pre,YBOCS_post,Unnamed: 110
456,sub-88012461,DISCOVERY,MDD,MDD,MDD-rTMS,YES,summer,NaN,1,1,...,3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
457,sub-88012461,DISCOVERY,MDD,MDD,MDD-rTMS,YES,fall,NaN,1,1,...,3,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
# save paths to eeg numpy files for each subject in df
ec_eeg_path, eo_eeg_path, has_ses1 = list(), list(), list()

for subj_id in df['participants_ID'].values:
    subj_path = f'{eeg_path}/{subj_id}/ses-1/eeg'
    if os.path.isdir(subj_path):
        has_ses1.append(True)
        subj_files = os.listdir(subj_path)
        ec_subj_files = list(pathlib.Path(subj_path).glob('*restEC*.npy'))
        eo_subj_files = list(pathlib.Path(subj_path).glob('*restEO*.npy'))
        assert len(ec_subj_files) == len(eo_subj_files) == 1

        ec_eeg_path.append(str(ec_subj_files[0]))
        eo_eeg_path.append(str(eo_subj_files[0]))

    else:
        has_ses1.append(False)
        ec_eeg_path.append('')
        eo_eeg_path.append('')

df['ec_eeg_path'] = ec_eeg_path
df['eo_eeg_path'] = eo_eeg_path
df['has_ses1'] = has_ses1

df = df[df['has_ses1'] == True].reset_index(drop=True)
print("Shape after removing subj with no session 1 data:", df.shape)

Shape after removing subj with no session 1 data: (128, 114)


## Bandpower

In [6]:
# define feature extraction function
def get_bandpower_features(subj_data_path, condition, normalize=True, pool_by_region=False):
    channel_filter = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC3', 'FCz', 'FC4',
                      'T7', 'C3', 'Cz', 'C4', 'T8', 'CP3', 'CPz', 'CP4', 'P7', 'P3',
                      'Pz', 'P4', 'P8', 'O1', 'Oz', 'O2',]
    
    regions = {
        'frontal': ['Fp1','Fp2','F7','F3','Fz','F4','F8'],
        'frontocentral': ['FC3','FCz','FC4'],
        'central': ['C3','Cz','C4','T7','T8'],
        'parietal': ['CP3','CPz','CP4','P7','P3','Pz','P4','P8'],
        'occipital': ['O1','Oz','O2']
    }

    eeg_dict = np.load(subj_data_path, allow_pickle=True)
    channel_labels = eeg_dict['labels']
    fs = eeg_dict['Fs']

    channel_mask = np.isin(channel_labels, channel_filter)
    eeg_data = eeg_dict['data'][0, channel_mask, :]
    # print(eeg_data.shape)
    assert eeg_data.shape[0] == len(channel_filter)

    bands = {
        'delta': [0.5, 4], 
        'theta': [4, 8], 
        'alpha': [8, 12], 
        'low_beta': [12, 20],
        'high_beta': [20, 30], 
        'low_gamma': [30, 40], 
        'high_gamma': [40, 80]
    }

    # construct dict with bandpower values for inputted subj
    feature_dict = {}
    channel_bandpower = {}

    for ch_idx, ch_name in enumerate(channel_filter):
        freqs, psd = compute_spectrum(eeg_data[ch_idx], fs, method='welch', avg_type='mean', nperseg=fs*2)
        
        band_powers = {}
        for band_name, (fmin, fmax) in bands.items():
            idx = (freqs >= fmin) & (freqs <= fmax)
            band_power = np.trapezoid(psd[idx], freqs[idx])
            band_powers[band_name] = band_power

        # normalize across all band powers for each channel
        if normalize:
            total_power = sum(band_powers.values()) + 1e-8
            for band_name in band_powers:
                band_powers[band_name] /= total_power

        channel_bandpower[ch_name] = band_powers
        # only save individual channel features if not pooling
        if not pool_by_region:
            for band_name, value in band_powers.items():
                feature_dict[f'{condition}_{ch_name}_{band_name}_power'] = value

    # pool by region if requested
    if pool_by_region:
        for region_name, region_channels in regions.items():
            for band_name in bands.keys():
                values = [channel_bandpower[ch][band_name] for ch in region_channels]
                feature_dict[f'{condition}_{region_name}_{band_name}_power'] = np.mean(values)

    return feature_dict

def extract_subj_features(subj_id, ec_path, eo_path):
    feats = {'participants_ID': subj_id}
    feats.update(get_bandpower_features(ec_path, 'EC', pool_by_region=True))
    feats.update(get_bandpower_features(eo_path, 'EO', pool_by_region=True))
    return feats

In [7]:
# extract features in parallel for all subjects
res = Parallel(n_jobs=16)(
    delayed(extract_subj_features)(subj_id, ec_path, eo_path)
    for subj_id, ec_path, eo_path in tqdm(
        zip(df['participants_ID'].values,
            df['ec_eeg_path'].values,
            df['eo_eeg_path'].values),
        total=len(df)
    )
)

  0%|          | 0/128 [00:00<?, ?it/s]

100%|██████████| 128/128 [00:02<00:00, 45.52it/s]


In [8]:
# save results in pickle file
bandpower_df = pd.DataFrame(res)
bandpower_df.to_pickle(f'../data/mdd_rtms_bandpower_pooled.pkl')
print("Shape of bandpower dataframe:", bandpower_df.shape)
print(bandpower_df.head())

Shape of bandpower dataframe: (128, 71)
  participants_ID  EC_frontal_delta_power  EC_frontal_theta_power  \
0    sub-87999321                0.378946                0.133278   
1    sub-88000181                0.199542                0.206217   
2    sub-88000313                0.203299                0.119568   
3    sub-88000489                0.234229                0.240737   
4    sub-88000533                0.281027                0.095075   

   EC_frontal_alpha_power  EC_frontal_low_beta_power  \
0                0.173084                   0.065740   
1                0.217618                   0.163459   
2                0.425803                   0.155984   
3                0.309522                   0.060355   
4                0.080683                   0.086534   

   EC_frontal_high_beta_power  EC_frontal_low_gamma_power  \
0                    0.070823                    0.045762   
1                    0.057527                    0.038488   
2                    0.04

## DynaMix Weights

In [6]:
# load the pre-trained model
model = load_hf_model("dynamix-6d-alrnn-v1.0")

# set model to evaluation mode
model.eval()

# initialize the forecaster
forecaster = DynaMixForecaster(model)

In [18]:
def get_dynamix_latent(subj_data_path):

    channel_filter = ['F7', 'P8', 'T7', 'O2']

    eeg_dict = np.load(subj_data_path, allow_pickle=True)
    channel_labels = eeg_dict['labels']
    sampling_freq = eeg_dict['Fs']

    channel_mask = np.isin(channel_labels, channel_filter)
    eeg_data = eeg_dict['data'][0, channel_mask, :]
    assert eeg_data.shape[0] == len(channel_filter)

    offset = 5000
    CL = 10000
    T = 100

    context_start = offset
    context_end = offset + CL

    # Load the time series data
    ts_data = eeg_data.T

    context_ts = ts_data[context_start:context_end,:] # context from 5000 to 15000 (10s-30s)

    # Convert to PyTorch tensor
    context_ts_tensor = torch.tensor(context_ts, dtype=torch.float32)
    
    activation = {}
    # a dict to store the activations
    def getActivation(name):
        activation[name] = list()
        # the hook signature
        def hook(model, input, output):
            activation[name].append(output.detach())
        return hook
    h = forecaster.model.gating_network.register_forward_hook(getActivation('w_exp'))

    # Make prediction
    with torch.no_grad():  # No gradient tracking needed for inference
        reconstruction_ts = forecaster.forecast(
            context=context_ts_tensor,
            horizon=T,
            preprocessing_method="pos_embedding",
            standardize=True,
            fit_nonstationary=False,
        )

    h.remove()
    w_exp = torch.stack(activation['w_exp']).numpy().squeeze()

    return w_exp

In [20]:
# extract features in parallel for all subjects
ec_res = Parallel(n_jobs=16)(delayed(get_dynamix_latent)(ec_path) for ec_path in tqdm(df['ec_eeg_path'].values))
eo_res = Parallel(n_jobs=16)(delayed(get_dynamix_latent)(ec_path) for ec_path in tqdm(df['eo_eeg_path'].values))









100%|██████████| 128/128 [00:08<00:00, 15.88it/s]





  0%|          | 0/128 [03:04<?, ?it/s]



100%|██████████| 128/128 [00:02<00:00, 43.41it/s]


In [21]:
print(np.array(ec_res).shape)
ec_res = np.array(ec_res)
eo_res = np.array(eo_res)
dynamix_latents = np.concatenate([eo_res, ec_res], axis=1)
print(dynamix_latents.shape)

(128, 100, 80)
(128, 200, 80)


In [29]:
# get ec, std result and save in pickle file
ec_std = np.std(ec_res, axis=1)

col_names = [f"dynamix_ec_std_{i+1}" for i in range(80)]
df_dynamix = pd.DataFrame(ec_std, columns=col_names)
df_dynamix.insert(0, "participants_ID", df['participants_ID'].values)

df_dynamix.to_pickle(f'../data/dynamix_ec_std_all.pkl')
print("Shape of dynamix dataframe:", df_dynamix.shape)
print(df_dynamix.head())

Shape of dynamix dataframe: (128, 81)
  participants_ID  dynamix_ec_std_1  dynamix_ec_std_2  dynamix_ec_std_3  \
0    sub-87999321      2.733999e-07          0.006910          0.033711   
1    sub-88000181      1.888021e-07          0.004220          0.017541   
2    sub-88000313      4.031916e-07          0.003680          0.011952   
3    sub-88000489      9.931152e-07          0.004275          0.014647   
4    sub-88000533      5.465943e-07          0.006813          0.029569   

   dynamix_ec_std_4  dynamix_ec_std_5  dynamix_ec_std_6  dynamix_ec_std_7  \
0          0.010207          0.005163      3.336415e-07          0.011144   
1          0.007555          0.004573      1.836598e-07          0.008980   
2          0.006222          0.011498      4.708596e-07          0.005364   
3          0.004476          0.002423      1.985160e-06          0.004443   
4          0.021561          0.009473      7.084412e-07          0.021621   

   dynamix_ec_std_8  dynamix_ec_std_9  ...  dyna